In [0]:
# Import all libraries used for multiple linear regression
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression

# Create Spark session
spark = SparkSession.builder.appName("supervised_ml").getOrCreate()


In [0]:
# Load the dataset into a Spark DataFrame
df1 = spark.read.format("csv").load("dbfs:/Workspace/Users/jessica.vargas006@mymdc.net/CAP4784 | Big Data/Linear_regression_dataset.csv",header="True",inferSchema="True")

In [0]:
print((df1.count(), len(df1.columns)))

(1232, 6)


In [0]:
df1.printSchema()

root
 |-- var_1: integer (nullable = true)
 |-- var_2: integer (nullable = true)
 |-- var_3: integer (nullable = true)
 |-- var_4: double (nullable = true)
 |-- var_5: double (nullable = true)
 |-- label: double (nullable = true)



In [0]:
# Check basic statistics of the variables
df1.show(10)

+-----+-----+-----+-----+-----+-----+
|var_1|var_2|var_3|var_4|var_5|label|
+-----+-----+-----+-----+-----+-----+
|  734|  688|   81|0.328|0.259|0.418|
|  700|  600|   94| 0.32|0.247|0.389|
|  712|  705|   93|0.311|0.247|0.417|
|  734|  806|   69|0.315| 0.26|0.415|
|  613|  759|   61|0.302| 0.24|0.378|
|  748|  676|   85|0.318|0.255|0.422|
|  669|  588|   97|0.315|0.251|0.411|
|  667|  845|   68|0.324|0.251|0.381|
|  758|  890|   64| 0.33|0.274|0.436|
|  726|  670|   88|0.335|0.268|0.422|
+-----+-----+-----+-----+-----+-----+
only showing top 10 rows


In [0]:
from pyspark.ml.linalg import Vector
from pyspark.ml.feature import VectorAssembler

In [0]:
df1.columns

['var_1', 'var_2', 'var_3', 'var_4', 'var_5', 'label']

In [0]:
# Create the features vector from var_1 to var_5 and keep label as target
vec_assmebler=VectorAssembler(inputCols=['var_1','var_2', 'var_3', 'var_4', 'var_5'],outputCol='features')

In [0]:
features_df1=vec_assmebler.transform(df1)

In [0]:
features_df1.printSchema()

root
 |-- var_1: integer (nullable = true)
 |-- var_2: integer (nullable = true)
 |-- var_3: integer (nullable = true)
 |-- var_4: double (nullable = true)
 |-- var_5: double (nullable = true)
 |-- label: double (nullable = true)
 |-- features: vectorudt (nullable = true)



In [0]:
# Split the data into training and test sets
train,test = features_df1.randomSplit([0.75, 0.25])
print(f"Size of train Dataset : {train.count()}" )

Size of train Dataset : 942


In [0]:
print(f"Size of test Dataset : {test.count()}" )

Size of test Dataset : 290


In [0]:
from pyspark.ml.regression import LinearRegression

In [0]:
# Train the original multiple linear regression model
lr = LinearRegression()

In [0]:
lr_model = lr.fit(train)

In [0]:
# Evaluate the original model on the test set
predictions_df=lr_model.transform(test)
predictions_df.show()

+-----+-----+-----+-----+-----+-----+--------------------+-------------------+
|var_1|var_2|var_3|var_4|var_5|label|            features|         prediction|
+-----+-----+-----+-----+-----+-----+--------------------+-------------------+
|  464|  640|   66|0.283| 0.22|0.301|[464.0,640.0,66.0...|0.31358767466394555|
|  486|  610|   61|0.293|0.233|0.332|[486.0,610.0,61.0...| 0.3187047001023693|
|  510|  588|   72|0.298|0.231|0.317|[510.0,588.0,72.0...|0.32335187431141366|
|  522|  621|   72|0.296|0.225|0.317|[522.0,621.0,72.0...| 0.3274887592791703|
|  527|  569|   75|0.297|0.239|0.341|[527.0,569.0,75.0...| 0.3333458629568357|
|  543|  615|   76|0.294|0.233|0.333|[543.0,615.0,76.0...|0.34035343631539194|
|  559|  613|   75|0.293|0.235|0.359|[559.0,613.0,75.0...| 0.3472181747383922|
|  567|  587|   84|0.301|0.238|0.349|[567.0,587.0,84.0...|0.34638636936180534|
|  569|  620|   77|0.302|0.247|0.349|[569.0,620.0,77.0...|0.35148280465921977|
|  570|  655|   66|0.311|0.246| 0.34|[570.0,655.0,66

In [0]:
model_predictions=lr_model.evaluate(test)
model_predictions.r2

0.8641063117631332

In [0]:
# Apply manual standard scaling to the features and retrain the model

from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

# List of original feature columns
feature_cols = ["var_1", "var_2", "var_3", "var_4", "var_5"]

# Compute mean and stddev for each feature
stats = df1.select(
    *[F.mean(c).alias(f"{c}_mean") for c in feature_cols],
    *[F.stddev(c).alias(f"{c}_std") for c in feature_cols]
).collect()[0]

# Create scaled versions of each feature: (x - mean) / std
df_scaled = df1
for c in feature_cols:
    m = stats[f"{c}_mean"]
    s = stats[f"{c}_std"]
    df_scaled = df_scaled.withColumn(f"{c}_scaled", (F.col(c) - m) / s)

scaled_feature_cols = [f"{c}_scaled" for c in feature_cols]

# Assemble the scaled features into a vector
assembler_scaled = VectorAssembler(
    inputCols=scaled_feature_cols,
    outputCol="scaled_features"
)

features_scaled_df = assembler_scaled.transform(df_scaled).select("scaled_features", "label")

# Train/test split on the scaled features
train_s, test_s = features_scaled_df.randomSplit([0.7, 0.3], seed=42)

# Train a new linear regression model on the scaled features
lr_scaled = LinearRegression(featuresCol="scaled_features", labelCol="label")
lr_scaled_model = lr_scaled.fit(train_s)

# Evaluate the scaled model
scaled_eval = lr_scaled_model.evaluate(test_s)

print("Original R2:", model_predictions.r2)      # from your earlier model
print("New R2 after manual scaling:", scaled_eval.r2)


/databricks/python/lib/python3.12/site-packages/pyspark/errors/exceptions/base.py:91: FutureWarning: Deprecated in 4.0.0, use getCondition instead.
  warnings.warn("Deprecated in 4.0.0, use getCondition instead.", FutureWarning)


---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-8631784265887668>, line 43
     40 # Evaluate the scaled model
     41 scaled_eval = lr_scaled_model.evaluate(test_s)
---> 43 print("Original R2:", model_predictions.r2)      # from your earlier model
     44 print("New R2 after manual scaling:", scaled_eval.r2)

File /databricks/python/lib/python3.12/site-packages/pyspark/ml/regression.py:633, in LinearRegressionSummary.r2(self)
    618 @property
    619 @since("2.0.0")
    620 def r2(self) -> float:
    621     """
    622     Returns R^2, the coefficient of determination.
    623 
   (...)
    631     <http://en.wikipedia.org/wiki/Coefficient_of_determination>`_
    632     """
--> 633     return self._call_java("r2")

File /databricks/python/lib/python3.12/site-packages/pyspark/ml/util.py:452, in try_remote_call.<locals>.wrapped(self, name, *args)
    449         retur